<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/detector_casco_obra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detector de casco en una obra – Laboratorio de Códigos
Entrena YOLO con el dataset **Construction-PPE** de Ultralytics (11 clases, incluye `helmet` y `no_helmet`) y lo aplica a un video:

- recuadro **verde** = con casco, recuadro **rojo** = sin casco
- cuenta cada persona sin casco **una sola vez** (seguimiento + confirmación de varios cuadros seguidos)

Activa GPU: Entorno de ejecución → Cambiar tipo → GPU. El dataset (178 MB, licencia AGPL-3.0) se descarga solo al entrenar.
Nota: sin probar con tus pesos reales. Si `yolo26n.pt` no existe en tu versión, usa `yolo11n.pt`.

In [1]:
!pip install -q -U ultralytics

In [2]:
import os, cv2, csv, subprocess, zipfile, numpy as np
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont

# ---------- CONFIGURACIÓN ----------
ENTRENAR       = True            # False si ya tienes pesos
EPOCAS         = 40
PESOS_PROPIOS  = None            # ej. "ppe/train/weights/best.pt" si ya entrenaste
VIDEO_IN       = "entrada/obra.mp4"   # sube tu video aquí (cámara fija y a la altura de la obra funciona mejor)
CONF           = 0.35
N_CONFIRMAR    = 8               # cuadros seguidos sin casco para contar una infracción (evita falsos positivos)
DIFUMINAR_CABEZAS = True         # pixela la zona de la cabeza (privacidad)
TITULO_1, TITULO_2 = "¿Quién no lleva", "casco en la obra?"
SALIDA         = "obra_sin_casco.mp4"
# -----------------------------------
os.makedirs("entrada", exist_ok=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
# Entrenamiento (o carga de pesos propios)
if PESOS_PROPIOS:
    ruta_pesos = PESOS_PROPIOS
elif ENTRENAR:
    try:
        base = YOLO("yolo26n.pt")
    except Exception:
        base = YOLO("yolo11n.pt")
    base.train(data="construction-ppe.yaml", epochs=EPOCAS, imgsz=640, batch=16, patience=10,
               project="ppe", name="train")
    ruta_pesos = str(base.trainer.best)          # ruta real de best.pt, sin adivinar carpetas
else:
    raise SystemExit("Pon ENTRENAR=True o indica PESOS_PROPIOS")
print("Pesos:", ruta_pesos)

modelo = YOLO(ruta_pesos)
m = modelo.val(data="construction-ppe.yaml", verbose=False)
nombres = modelo.names
for i, ap in zip(m.box.ap_class_index, m.box.ap50):
    if nombres[int(i)] in ("helmet", "no_helmet"):
        print(f"AP50 {nombres[int(i)]}: {ap:.3f}")
print("mAP50 (todas las clases):", round(float(m.box.map50), 3))

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=construction-ppe.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto

In [4]:
# Utilidades
W, H, FPS_OUT = 1080, 1920, 30
ID_CASCO = [k for k, v in modelo.names.items() if v == "helmet"][0]
ID_SIN   = [k for k, v in modelo.names.items() if v == "no_helmet"][0]
VERDE, ROJO = (60, 200, 60), (40, 40, 255)

def _fuente(s):
    for f in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
        if os.path.exists(f):
            return ImageFont.truetype(f, s)
    return ImageFont.load_default()

def detectar(frame):
    # Devuelve (cajas xyxy, clases, confianzas, ids) solo de helmet / no_helmet
    r = modelo.track(frame, persist=True, conf=CONF, tracker="bytetrack.yaml", verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return np.zeros((0, 4)), np.zeros(0, int), np.zeros(0), np.zeros(0, int)
    xyxy = r.boxes.xyxy.cpu().numpy()
    cls = r.boxes.cls.cpu().numpy().astype(int)
    conf = r.boxes.conf.cpu().numpy()
    ids = r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else np.full(len(cls), -1)
    k = np.isin(cls, [ID_CASCO, ID_SIN])
    return xyxy[k], cls[k], conf[k], ids[k]

def pixelar(img, x1, y1, x2, y2):
    h_, w_ = img.shape[:2]
    x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, w_), min(y2, h_)
    if x2 - x1 < 4 or y2 - y1 < 4:
        return
    roi = img[y1:y2, x1:x2]
    peq = cv2.resize(roi, (max(2, (x2 - x1) // 12), max(2, (y2 - y1) // 12)))
    img[y1:y2, x1:x2] = cv2.resize(peq, (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)

def dibujar(frame, xyxy, cls, conf):
    fh = frame.shape[0]; grosor = max(2, fh // 270); esc = max(0.5, fh / 900)
    if DIFUMINAR_CABEZAS:                                 # primero la privacidad
        for (x1, y1, x2, y2) in xyxy.astype(int):
            bw, bh = x2 - x1, y2 - y1
            pixelar(frame, x1 - bw // 6, y1 - bh // 8, x2 + bw // 6, y2 + int(bh * 0.9))
    for (x1, y1, x2, y2), c_, p in zip(xyxy.astype(int), cls, conf):
        col = ROJO if c_ == ID_SIN else VERDE
        txt = "SIN CASCO" if c_ == ID_SIN else "CASCO"
        cv2.rectangle(frame, (x1, y1), (x2, y2), col, grosor)
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, esc, grosor)
        cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 8, y1), col, -1)
        cv2.putText(frame, txt, (x1 + 4, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, esc, (255, 255, 255), grosor)
    return frame

In [5]:
# Si no subiste video, crea uno de DEMO con imágenes de prueba del dataset (solo para ver que todo funciona)
if not os.path.exists(VIDEO_IN):
    from ultralytics.utils import DATASETS_DIR
    imgs = sorted([str(p) for p in (DATASETS_DIR / "construction-ppe" / "images" / "test").glob("*.jpg")])[:10]
    wr = cv2.VideoWriter(VIDEO_IN, cv2.VideoWriter_fourcc(*"mp4v"), 12, (1280, 720))
    for p in imgs:
        im = cv2.imread(p); s = min(1280 / im.shape[1], 720 / im.shape[0])
        im = cv2.resize(im, (int(im.shape[1] * s), int(im.shape[0] * s)))
        lienzo = np.zeros((720, 1280, 3), np.uint8)
        lienzo[(720 - im.shape[0]) // 2:(720 - im.shape[0]) // 2 + im.shape[0],
               (1280 - im.shape[1]) // 2:(1280 - im.shape[1]) // 2 + im.shape[1]] = im
        for _ in range(18):
            wr.write(lienzo)
    wr.release()
    print("Video DEMO creado (sube tu propio video para el resultado real).")

Video DEMO creado (sube tu propio video para el resultado real).


In [8]:
cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    print(f"Advertencia: No se pudo abrir el video {VIDEO_IN}. Asegúrate de que el archivo existe y es válido.")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Evitar división por cero si el video no se cargó correctamente
if fw == 0 or fh == 0:
    fw, fh = 1280, 720

esc = min(W / fw, 1000 / fh); pw, ph = int(fw * esc), int(fh * esc)
x0, y0 = (W - pw) // 2, 520 + (1000 - ph) // 2

bg = Image.new("RGB", (W, H), (9, 9, 16)); d = ImageDraw.Draw(bg)
for txt, y, col in [(TITULO_1, 170, (255, 255, 255)), (TITULO_2, 275, (255, 214, 10))]:
    f = _fuente(78); bb = d.textbbox((0, 0), txt, font=f)
    d.text(((W - (bb[2] - bb[0])) // 2, y), txt, font=f, fill=col, stroke_width=3, stroke_fill=(0, 0, 0))
bg = np.array(bg)[:, :, ::-1].copy()

racha, infractores, eventos = {}, set(), []         # eventos: (segundo, id)
flash_hasta = -1.0
raw = "_raw_obra.mp4"
wr = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*"mp4v"), FPS_OUT, (W, H))
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    t = i / fps
    xyxy, cls, conf, ids = detectar(frame)

    sin_ahora = set(int(k) for k, c_ in zip(ids, cls) if c_ == ID_SIN and k != -1)
    for k in sin_ahora:
        racha[k] = racha.get(k, 0) + 1
        if racha[k] >= N_CONFIRMAR and k not in infractores:
            infractores.add(k); eventos.append((round(t, 2), k)); flash_hasta = t + 0.6
    for k in list(racha):
        if k not in sin_ahora:
            racha[k] = 0

    frame = dibujar(frame, xyxy, cls, conf)
    canvas = bg.copy()
    canvas[y0:y0 + ph, x0:x0 + pw] = cv2.resize(frame, (pw, ph), interpolation=cv2.INTER_CUBIC)
    if t < flash_hasta:                               # borde rojo de alerta
        cv2.rectangle(canvas, (x0, y0), (x0 + pw, y0 + ph), ROJO, 14)
    pil = Image.fromarray(canvas[:, :, ::-1]); dd = ImageDraw.Draw(pil)
    dd.rounded_rectangle((x0 + 20, y0 + 20, x0 + 20 + 520, y0 + 110), 16, fill=(30, 30, 230))
    dd.text((x0 + 44, y0 + 30), f"SIN CASCO: {len(infractores)}", font=_fuente(52), fill=(255, 255, 255))
    if t < flash_hasta:
        txt = "ALERTA: SIN CASCO"; f = _fuente(60); bb = dd.textbbox((0, 0), txt, font=f)
        dd.text(((W - (bb[2] - bb[0])) // 2, y0 + ph + 50), txt, font=f, fill=(255, 70, 70), stroke_width=3, stroke_fill=(0, 0, 0))
    out = np.array(pil)[:, :, ::-1]
    for _ in range(max(1, round(FPS_OUT / fps))):
        wr.write(out)
    i += 1
cap.release(); wr.release()
if i > 0:
    subprocess.run(f"ffmpeg -y -loglevel error -i {raw} -c:v libx264 -pix_fmt yuv420p -crf 18 -movflags +faststart {SALIDA}", shell=True, check=True)
    os.remove(raw)
    print("Video listo:", SALIDA, "| personas sin casco contadas:", len(infractores), "| duración:", round(i / fps, 1), "s")
else:
    print("No se procesaron fotogramas. Verifica que el archivo de entrada sea un video válido.")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 287ms
Prepared 1 package in 70ms
Installed 1 package in 1ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Video listo: obra_sin_casco.mp4 | personas sin casco contadas: 18 | duración: 30.0 s
